In [75]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter

In [99]:
data = pd.read_csv('train.csv')

In [81]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  3800 non-null   object 
 1   admission_date              3800 non-null   object 
 2   age                         3800 non-null   float64
 3   gender                      3800 non-null   object 
 4   admission_type              3800 non-null   int64  
 5   discharge_destination       3800 non-null   int64  
 6   discharge_day_of_week       3800 non-null   object 
 7   length_of_stay_days         3800 non-null   float64
 8   charlson_comorbidity_index  3800 non-null   float64
 9   prior_admissions_1yr        3800 non-null   float64
 10  n_medications_discharge     3800 non-null   int64  
 11  insurance_type              3800 non-null   object 
 12  glucose_level_mgdl          3127 non-null   float64
 13  blood_pressure_systolic     3800 

In [78]:
data.describe()

,age,admission_type,discharge_destination,length_of_stay_days,charlson_comorbidity_index,prior_admissions_1yr,n_medications_discharge,glucose_level_mgdl,blood_pressure_systolic,sodium_meql,creatinine_mgdl,haemoglobin_gdl,readmitted_30d
count,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3127.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000
mean,56.566842,1.542105,1.656579,7.247921,2.268421,2.152105,9.699737,104.443780,114.079995,137.855211,1.199868,12.471132,0.090000
std,56.968099,0.658486,0.923051,4.816102,2.271643,2.133976,5.768190,26.450105,40.952359,3.992365,0.500145,1.975551,0.286219
min,18.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,60.000000,10.670000,125.000000,0.400000,6.000000,0.000000
25%,35.000000,1.000000,1.000000,4.000000,1.000000,0.000000,5.000000,85.100000,106.300000,135.100000,0.860000,11.100000,0.000000
50%,53.000000,1.000000,1.000000,6.000000,2.000000,2.000000,10.000000,103.600000,124.200000,137.700000,1.110000,12.500000,0.000000
75%,72.000000,2.000000,2.000000,9.100000,3.000000,3.000000,15.000000,122.400000,138.800000,140.500000,1.430000,13.800000,0.000000
max,999.000000,3.000000,4.000000,45.100000,10.000000,11.000000,19.000000,188.000000,209.800000,154.300000,4.910000,18.000000,1.000000


In [100]:
data.drop(['patient_id','admission_date','discharge_day_of_week'],axis=1,inplace=True)

In [101]:
X = data.drop('readmitted_30d', axis=1)
y = data['readmitted_30d']

# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing Steps

## Observation
*   glucose_level_mgdl contains 673 null values. The missing is random as distribution of missingness across all other variable is same and the auc value is 0.5. Hence missingness is random.



## Actions
*   Remove patient Id
*   Remove admission date
*   Remove discharge_day_of_week
*   label encoding of gender, insurance_type



In [102]:
#preprocessing
columns_to_encode = ['gender', 'insurance_type']
le_dict = {}

for col in columns_to_encode:
    le = LabelEncoder()
    le_dict[f"{col}_encoder"] = le
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])


X_train['age'] = X_train['age'].replace(999, np.nan)
mean_age = X_train['age'].mean()
X_train['age'].fillna(mean_age, inplace=True)
X_test['age'].fillna(mean_age, inplace=True)

gender_glucose_median = X_train.groupby('gender')['glucose_level_mgdl'].median()
X_train['glucose_level_mgdl'] = X_train['glucose_level_mgdl'].fillna(X_train['gender'].map(gender_glucose_median))
X_test['glucose_level_mgdl'] = X_test['glucose_level_mgdl'].fillna(X_test['gender'].map(gender_glucose_median))


smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

/tmp/ipykernel_8086/2662161763.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train['age'].fillna(mean_age, inplace=True)
/tmp/ipykernel_8086/2662161763.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [95]:
le_dict

{'gender_encoder': LabelEncoder(), 'insurance_type_encoder': LabelEncoder()}

### Saving the Data Preprocessing Pipeline Components

In [97]:
import pickle
from sklearn.preprocessing import LabelEncoder

# Re-initialize and fit LabelEncoders on the full 'X' data before splitting/encoding
# This ensures all possible categories are captured for deployment
gender_encoder = LabelEncoder()
gender_encoder.fit(X['gender'])

insurance_type_encoder = LabelEncoder()
insurance_type_encoder.fit(X['insurance_type'])

# Collect all preprocessing components into a dictionary
preprocessing_pipeline_components = {
    'gender_encoder': le_dict['gender_encoder'],
    'insurance_type_encoder': le_dict['insurance_type_encoder'],
    'mean_age': mean_age, # Already calculated from X_train
    'gender_glucose_median': gender_glucose_median # Already calculated from X_train
}

# Define the path to save the preprocessing pipeline components
pipeline_save_path = 'preprocessing_pipeline.pkl'

# Save the preprocessing pipeline components
with open(pipeline_save_path, 'wb') as f:
    pickle.dump(preprocessing_pipeline_components, f)

print(f"Preprocessing pipeline components saved to {pipeline_save_path}")

Preprocessing pipeline components saved to preprocessing_pipeline.pkl


# Model Training

In [103]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Convert pandas DataFrames/Series to PyTorch Tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # Unsqueeze to make it (batch_size, 1)
X_train_resampled_tensor = torch.tensor(X_train_resampled.values, dtype=torch.float32)
y_train_resampled_tensor = torch.tensor(y_train_resampled.values, dtype=torch.float32).unsqueeze(1)

# Create a TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_resampled_dataset = TensorDataset(X_train_resampled_tensor, y_train_resampled_tensor)

# Create a DataLoader
batch_size = 64
train_loader = DataLoader(train_resampled_dataset, batch_size=batch_size, shuffle=True)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")
print(f"Number of batches in DataLoader: {len(train_loader)}")

Using device: cuda
X_train_tensor shape: torch.Size([3040, 14])
y_train_tensor shape: torch.Size([3040, 1])
Number of batches in DataLoader: 87


In [104]:
class SimpleANN(nn.Module):
    def __init__(self, input_size):
        super(SimpleANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 8) # First hidden layer with 4 neurons
        self.bn1 = nn.BatchNorm1d(8)       # Batch Normalization for first hidden layer
        self.relu = nn.ReLU()              # Activation function
        self.dropout = nn.Dropout(0.5)     # Dropout layer with 50% dropout rate

        self.fc2 = nn.Linear(8, 4)         # Second hidden layer with 2 neurons
        self.bn2 = nn.BatchNorm1d(4)       # Batch Normalization for second hidden layer
        self.relu2 = nn.ReLU()             # Activation function

        self.fc3 = nn.Linear(4, 1)         # Output layer for binary classification
        self.sigmoid = nn.Sigmoid()        # Added Sigmoid activation for binary output

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x) # Apply dropout after first ReLU

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)

        x = self.fc3(x)
        x = self.sigmoid(x) # Apply sigmoid to the output
        return x

# Instantiate the model with the new architecture
input_size = X_train.shape[1]
model = SimpleANN(input_size).to(device)

# Define Loss function and Optimizer
criterion = nn.BCELoss() # Changed to BCELoss because Sigmoid is now part of the model
optimizer = optim.RMSprop(model.parameters(), lr=0.0001) # Changed optimizer to RMSprop

print("ANN Model Architecture:")
print(model)

ANN Model Architecture:
SimpleANN(
  (fc1): Linear(in_features=14, out_features=8, bias=True)
  (bn1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (bn2): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [ ]:
# Training the model
num_epochs = 200

print("\nStarting training...")
for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        output = model(data)

        # Calculate loss
        loss = criterion(output, target)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training finished.")

In [108]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert test data to PyTorch Tensors
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Create a TensorDataset and DataLoader for the test set
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
batch_size = 64 # Use the same batch size as training
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # No need to shuffle test data

# Evaluate the model
model.eval() # Set the model to evaluation mode
all_predictions = []
all_targets = []
all_probabilities = []

with torch.no_grad(): # Disable gradient calculation during evaluation
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        all_probabilities.extend(output.cpu().numpy().flatten()) # Store raw probabilities
        predictions = (torch.sigmoid(output) > 0.7).float() # Apply sigmoid and threshold for binary prediction
        all_predictions.extend(predictions.cpu().numpy().flatten())
        all_targets.extend(target.cpu().numpy().flatten())

# Convert to numpy arrays for easier analysis
all_probabilities = np.array(all_probabilities)

# Print statistics of raw probabilities
print(f"\nRaw Output Logits (min, max, mean):\n  Min: {all_probabilities.min():.4f}\n  Max: {all_probabilities.max():.4f}\n  Mean: {all_probabilities.mean():.4f}")

# Calculate metrics
accuracy = accuracy_score(all_targets, all_predictions)
precision = precision_score(all_targets, all_predictions)
recall = recall_score(all_targets, all_predictions)

print(f"\nModel Evaluation on Test Set (Threshold = 0.7):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")



Raw Output Logits (min, max, mean):
  Min: 0.0040
  Max: 0.9977
  Mean: 0.2169

Model Evaluation on Test Set (Threshold = 0.7):
Accuracy: 0.9329
Precision: 0.6197
Recall: 0.6471


### Data Preprocessing Pipeline Function

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(df_raw, preprocessing_pipeline_path='preprocessing_pipeline.pkl'):
    """
    Applies the saved preprocessing steps to a raw DataFrame.

    Args:
        df_raw (pd.DataFrame): The raw DataFrame to preprocess.
        preprocessing_pipeline_path (str): Path to the saved preprocessing pipeline components.

    Returns:
        pd.DataFrame: The preprocessed DataFrame.
    """
    # Load the preprocessing components
    with open(preprocessing_pipeline_path, 'rb') as f:
        components = pickle.load(f)

    gender_encoder = components['gender_encoder']
    insurance_type_encoder = components['insurance_type_encoder']
    mean_age = components['mean_age']
    gender_glucose_median = components['gender_glucose_median']

    # Make a copy to avoid modifying the original DataFrame
    df = df_raw.copy()

    # Drop unnecessary columns if they exist
    columns_to_drop = ['patient_id', 'admission_date', 'discharge_day_of_week']
    for col in columns_to_drop:
        if col in df.columns:
            df = df.drop(columns=[col])

    # Apply Label Encoding
    if 'gender' in df.columns:
        df['gender'] = gender_encoder.transform(df['gender'])
    if 'insurance_type' in df.columns:
        df['insurance_type'] = insurance_type_encoder.transform(df['insurance_type'])

    # Handle 'age' column
    if 'age' in df.columns:
        df['age'] = df['age'].replace(999, np.nan)
        df['age'].fillna(mean_age, inplace=True)

    # Handle 'glucose_level_mgdl' missing values
    if 'glucose_level_mgdl' in df.columns and df['glucose_level_mgdl'].isnull().any():
        if 'gender' in df.columns:
            df['glucose_level_mgdl'] = df['glucose_level_mgdl'].fillna(df['gender'].map(gender_glucose_median))
        else:
            # Fallback if gender is somehow not available for glucose imputation
            df['glucose_level_mgdl'].fillna(df['glucose_level_mgdl'].median(), inplace=True)

    # Ensure the order of columns matches the training data (important for ANN)
    # This assumes X_train's columns are representative of the expected order
    # If X_train isn't available, we'd need to store the column order in the pipeline components.
    # For now, let's assume the current X_train's columns variable is accessible and correctly ordered
    global X_train # Access the global X_train DataFrame
    if 'X_train' in globals() and isinstance(X_train, pd.DataFrame):
        # Align columns, adding missing ones with 0 and dropping extra ones
        missing_cols = set(X_train.columns) - set(df.columns)
        for c in missing_cols:
            df[c] = 0
        df = df[X_train.columns]
    else:
        print("Warning: X_train not found in global scope. Cannot ensure column order for preprocessing.")

    return df

### Prediction on New Data (test.csv)

In [ ]:
import pandas as pd
import torch

# Assuming test.csv is available in the current directory
try:
    test_df_raw = pd.read_csv('test.csv')
    print("test.csv loaded successfully.")
except FileNotFoundError:
    print("Error: 'test.csv' not found. Please ensure the file is in the correct directory.")
    # Create a dummy test_df_raw for demonstration if test.csv is not present
    # In a real scenario, you would handle this file not found error more robustly
    test_df_raw = data.sample(5).copy() # Using 5 random rows from 'data' as dummy test data
    test_df_raw = test_df_raw.drop('readmitted_30d', axis=1, errors='ignore')
    print("Using dummy data for demonstration as test.csv was not found.")

# Display the raw test data head
print("\nRaw Test Data Head:")
display(test_df_raw.head())

In [ ]:
# Preprocess the raw test data using the pipeline function
print("\nPreprocessing test data...")
preprocessed_test_df = preprocess_data(test_df_raw, preprocessing_pipeline_path='preprocessing_pipeline.pkl')

print("Preprocessed Test Data Head:")
display(preprocessed_test_df.head())

print("Checking for missing values in preprocessed test data:")
print(preprocessed_test_df.isnull().sum())

In [ ]:
# Load the trained model

# Re-instantiate the model with the same architecture as trained
input_size = preprocessed_test_df.shape[1]
loaded_model = SimpleANN(input_size).to(device)

# Load the saved state dictionary
model_save_path = 'simple_ann_model.pth'
loaded_model.load_state_dict(torch.load(model_save_path))
loaded_model.eval() # Set to evaluation mode

print("Model loaded successfully.")
print("Loaded Model Architecture:")
print(loaded_model)

In [ ]:
# Make predictions
print("\nMaking predictions...")

# Convert preprocessed test data to PyTorch tensor
test_tensor = torch.tensor(preprocessed_test_df.values, dtype=torch.float32).to(device)

with torch.no_grad():
    raw_outputs = loaded_model(test_tensor)
    # Apply the same threshold as used in evaluation (0.7)
    binary_predictions = (raw_outputs > 0.7).float().cpu().numpy()

# Convert binary predictions to 'Yes'/'No'
final_predictions = pd.Series(binary_predictions.flatten()).map({1.0: 'Yes', 0.0: 'No'})

print("Predictions completed. Displaying first 10 predictions:")
display(final_predictions.head(10))

# You can also add predictions back to a DataFrame for easier viewing
predictions_df = test_df_raw.copy()
predictions_df['readmitted_30d_prediction'] = final_predictions.values

print("\nTest Data with Predictions:")
display(predictions_df.head())

### Saving the Trained PyTorch Model

In [107]:
import torch

# Define the path to save the model
model_save_path = 'simple_ann_model.pth'

# Save the model's state dictionary
torch.save(model.state_dict(), model_save_path)

print(f"Trained PyTorch model saved to {model_save_path}")

Trained PyTorch model saved to simple_ann_model.pth
